# Gearbox failure prediction — results

**Author:** a colleague who has left the company
**Status:** ready for deployment, pending sign-off

We predict which wind-turbine gearboxes will need an unplanned repair in the
coming quarter, from weekly vibration and temperature telemetry plus the
turbine's installation record.

**Headline result: area under the receiver operating characteristic (ROC) curve (AUC) of 0.92 under 5-fold cross-validation.**

The maintenance team would like to start scheduling from this next month. Your
job is to sign it off — or not.

---

*This notebook runs. Every number in it is real. It is also wrong, in more than
one way. Do not trust anything below because it looks careful.*

In [1]:
# pip install numpy==1.26.4 pandas==2.1.1 matplotlib==3.8.0 scikit-learn==1.3.1

import numpy as np
import pandas as pd

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from turbine_data import ALL_COLUMNS, load_turbines

data = load_turbines()
print(f"{len(data)} readings")
print(f"{data['needs_repair'].mean():.0%} of readings are labelled 'needs repair'")
data.head()

480 readings
50% of readings are labelled 'needs repair'


,turbine,week,vibration_rms,oil_temperature_c,particle_count,power_output_kw,ambient_humidity,hub_height_m,rotor_diameter_m,site_elevation_m,commissioning_year,gearbox_ratio,blade_batch,foundation_depth_m,grid_distance_km,needs_repair
0,T00,1,2.28,54.65,25,1900.40,69.77,110.0,100.0,492.0,2011.0,97.0,7.0,4.5,11.8,1
1,T00,2,2.60,63.10,14,1669.70,85.58,110.0,100.0,492.0,2011.0,97.0,7.0,4.5,11.8,1
2,T00,3,2.31,59.95,39,2053.99,43.75,110.0,100.0,492.0,2011.0,97.0,7.0,4.5,11.8,1
3,T00,4,2.81,64.35,33,1972.62,87.18,110.0,100.0,492.0,2011.0,97.0,7.0,4.5,11.8,1
4,T00,5,2.84,68.04,35,1772.54,73.38,110.0,100.0,492.0,2011.0,97.0,7.0,4.5,11.8,1


## Features

We use the weekly telemetry together with the installation record — hub height,
rotor diameter, commissioning year and so on. The installation record is
strongly associated with the outcome in our data, so it clearly carries
signal.

In [2]:
X = data[ALL_COLUMNS]
y = data["needs_repair"]

print(f"{X.shape[1]} features, {len(X)} rows")
list(X.columns)

13 features, 480 rows


['vibration_rms',
 'oil_temperature_c',
 'particle_count',
 'power_output_kw',
 'ambient_humidity',
 'hub_height_m',
 'rotor_diameter_m',
 'site_elevation_m',
 'commissioning_year',
 'gearbox_ratio',
 'blade_batch',
 'foundation_depth_m',
 'grid_distance_km']

## A quick check that scaling helps

We standardise the features before fitting, as recommended in lesson 2.

In [3]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

print("means after scaling:", np.round(X_scaled.mean().values[:4], 3))
print("sds after scaling:  ", np.round(X_scaled.std().values[:4], 3))

means after scaling: [ 0. -0. -0. -0.]
sds after scaling:   [1.001 1.001 1.001 1.001]


## Keeping the most informative features

With this many columns it is sensible to keep only the ones that carry signal,
so we score every feature against the target and keep the best eight.

In [4]:
selector = SelectKBest(f_classif, k=8).fit(X_scaled, y)
X_selected = X_scaled.loc[:, selector.get_support()]

print("kept:", list(X_selected.columns))

kept: ['vibration_rms', 'oil_temperature_c', 'particle_count', 'power_output_kw', 'hub_height_m', 'rotor_diameter_m', 'site_elevation_m', 'blade_batch']


## The result

Five-fold cross-validation, stratified so each fold keeps the class balance.

In [5]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(LogisticRegression(max_iter=5_000), X_selected, y,
                         cv=folds, scoring="roc_auc")

print(f"fold scores: {np.round(scores, 3)}")
print(f"\nAUC = {scores.mean():.3f} +/- {scores.std():.3f}")

fold scores: [0.944 0.916 0.91  0.927 0.922]

AUC = 0.924 +/- 0.012


## Conclusion

**AUC 0.92**, with tight agreement between the folds and a clear margin over the
0.5 baseline. The model is ready for deployment.

Recommended next step: schedule inspections for the turbines the model scores
above 0.5, starting next month.

---

### Your job

Do not accept this. Work out what is wrong with it — there is more than one
thing — and produce the number that should have been reported.

See `05_experimental_methodology.md` for what to hand in.